In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
!pip install -q scipy
!pip install -q h5py
!pip install -q matplotlib
!pip install -q torch==1.13.1+cu117 torchvision==0.14.1+cu117 torchaudio==0.13.1 --extra-index-url https://download.pytorch.org/whl/cu117
!pip install -q transformers
!pip install -q fuzzy_match
!pip install -q nltk
!pip install -q rouge
!pip install -q diffusers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 GB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.3/24.3 MB 66.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 93.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchdata 0.7.0 requires torch==2.1.0, but you have torch 1.13.1+cu117 which is incompatible.
torchtext 0.16.0 requires torch==2.1.0, but you have torch 1.13.1+cu117 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.7 MB/s eta 0:00:00


In [ ]:
%cd /content/drive/MyDrive/Research/FINAL/Code/MMMM

/content/drive/MyDrive/Research/FINAL/Code/MMMM


In [ ]:
import sys
sys.path.append("./models")
sys.path.append("./training")
sys.path.append("./testing")
sys.path.append("./utils")
sys.path.append("./ZuCo")
sys.path.append("./trainer")

import torch
import torch.nn as nn
import torch.optim as optim

from master_init import *
from DSG import *

In [ ]:
config = {
    "device" : "cuda:0",
    "device_ids" : [0]
}
device = config["device"]
device_ids = config["device_ids"]

model = INITIALIZE_MODEL(device=device, device_ids=device_ids).to(device)

config.json:   0%|          | 0.00/4.52k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/961k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

In [ ]:
dataset_dict = INITIALIZE_DATALOADERS(
    keys=["Brain2Image"],
    bsz=[1]
)

In [ ]:
dsg_tasks = DSGTasks()
dsg_tasks.add_task(
    DSGTask(
        task_name="EEG-IMG-CLASSIFICATION",
        dataset_tag="Brain2Image",
        criterion=nn.CrossEntropyLoss(), # CE Loss for Tenary Sentiment
        optimizer=optim.Adam,
        learning_rate=5e-3,
        converge_lim=2,
        converge_threshold=0.005,
        div_threshold=0.01
        )
    )

## Code

In [ ]:
# # dataloader = args_dict["dataloader"]
# # model = args_dict["model"]
# # optimizer = args_dict["optimizer"]
# # tokenizer = args_dict["tokenizer"]
# # criterion = args_dict["criterion"]
# # device = args_dict["device"] if "device" in args_dict else "cuda"
# # device_ids = args_dict["device_ids"] if "device_ids" in args_dict else None
# # staging_device = args_dict["staging_device"] if "staging_device" in args_dict else None

# dataloader = dataset_dict["Brain2Image"]
# model = model
# optimizer = optim.Adam(model.parameters(), lr=5e-3)
# from transformers import BartTokenizer
# criterion = nn.CrossEntropyLoss()
# device = "cuda"
# device_ids = None
# staging_device = "cuda"

In [ ]:
# running_loss = 0.0
# tot_cnt = 0

# phase="train"
# # Iterate over data.
# current_data = dataloader[phase].load_data()
# while not current_data["reset"]:
#     eegs = current_data["data"]
#     expected = current_data["classes"]
#     target = torch.zeros(eegs.shape[0], 40).to(device, dtype=float)
#     for i in range(len(expected)):
#         target[i][expected[i]] = 1.0

#     optimizer.zero_grad()

#     args_dict = {
#         "input_data_batch" : eegs.to(device),
#         "pool_result" : True
#         }

#     output = model(
#         mode="EEG-IMG-BRAIN2IMAGE-CLASSIFICATION",
#         args_dict=args_dict,
#         staging_device=staging_device
#         )

#     loss = criterion(output.to(dtype=float), target.to(dtype=float))

#     # Backward + Optimize only if in training phase
#     if phase == 'train':
#         if device_ids == None:
#             loss.backward()
#             optimizer.step()
#         else:
#             loss.mean().backward()
#             optimizer.step()

#     # Compute stats
#     if device_ids == None:
#         running_loss += loss.item() * eegs.size()[0]
#     else:
#         running_loss += loss.mean().item() * eegs.size()[0]
#     tot_cnt += eegs.size()[0]
#     current_data = dataloader[phase].load_data()

# epoch_loss = running_loss / tot_cnt

## Training Code

In [ ]:
import EEG_IMG_CLASSIFICATION
from tqdm import tqdm

In [ ]:
dataloader = dataset_dict["Brain2Image"]
model = model
optimizer = optim.Adam(model.parameters(), lr=5e-4)
criterion = nn.CrossEntropyLoss()
device = "cuda"
device_ids = None

In [ ]:
num_epochs = 10
for i in tqdm(range(num_epochs)):
    args_dict = {
        "dataloader" : dataloader,
        "model" : model,
        "optimizer" : optimizer,
        "criterion" : criterion,
        "device" : device,
        "device_ids" : None,
        "staging_device" : "cuda"
    }
    results = EEG_IMG_CLASSIFICATION.train(args_dict)
    model = results["model"]
    print(i, results["train_loss"], results["dev_loss"])


 11%|█         | 1/9 [04:51<38:50, 291.35s/it]

1 3.6888947426748455 3.688985103981046


 22%|██▏       | 2/9 [10:19<36:31, 313.10s/it]

2 3.6888761425484575 3.689029449733453


 33%|███▎      | 3/9 [15:49<32:03, 320.65s/it]

...

7 3.6888031049064214 3.689247416785288


 89%|████████▉ | 8/9 [43:14<05:28, 328.17s/it]

8 3.6887899043570003 3.689290021393224


100%|██████████| 9/9 [48:43<00:00, 324.85s/it]

9 3.688777178512271 3.6893322075090977


In [ ]:
torch.save(model.state_dict(), "./BUF_EEG-IMG-BRAIN2IMAGE-CLASSIFICATION.pt")

In [ ]:
import numpy as np

In [ ]:
np.zeros((40, 40))

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [ ]:
def evaluate(args_dict):
    dataloader = args_dict["dataloader"]
    model = args_dict["model"]
    device = args_dict["device"] if "device" in args_dict else "cuda"
    device_ids = args_dict["device_ids"] if "device_ids" in args_dict else None
    staging_device = args_dict["staging_device"] if "staging_device" in args_dict else None
    if staging_device==None:
        staging_device = f"cuda:{device_ids[0]}" if device_ids == None else "cuda"
    results = {}
    for phase in ['train', 'dev']:
        if phase == 'train':
            model.train()    # Set model to training mode
        else:
            model.eval()     # Set model to evaluate mode

        tot_cnt = 0
        confusion_matrix = np.zeros((40, 40))

        # Iterate over data.
        current_data = dataloader[phase].load_data()
        while not current_data["reset"]:
            eegs = current_data["data"]
            expected = current_data["classes"]

            model.zero_grad()

            args_dict = {
                "input_data_batch" : eegs.to(device),
                "pool_result" : True
                }

            output = model(
                mode="EEG-IMG-BRAIN2IMAGE-CLASSIFICATION",
                args_dict=args_dict,
                staging_device=staging_device
                )

            for i in range(output.shape[0]):
                confusion_matrix[expected[i]][np.argmax(output[i].cpu().detach().numpy())] += 1

            tot_cnt += eegs.size()[0]
            current_data = dataloader[phase].load_data()

        results[f"{phase}_confusion_matrix"] = confusion_matrix
        tp = np.array([confusion_matrix[i][i] for i in range(40)])
        fp = np.array([confusion_matrix[i,:].sum() - confusion_matrix[i][i] for i in range(40)]) - tp
        fn = np.array([confusion_matrix[:,i].sum() - confusion_matrix[i][i] for i in range(40)]) - tp
        tn = np.array([tot_cnt - tp[i] - fp[i] - fn[i] for i in range(40)])
        results[f"{phase}_TP"] = tp
        results[f"{phase}_FP"] = fp
        results[f"{phase}_TN"] = tn
        results[f"{phase}_FN"] = fn
        results[f"{phase}_accuracy"] = accuracy = (tp + tn) / tot_cnt
        results[f"{phase}_precision"] = precision = tp / (tp + fp)
        results[f"{phase}_recall"] = recall = tp / (tp + fn)
        results[f"{phase}_f1"] = f1 = 2 * precision * recall / (precision + recall)
    model.zero_grad()

    return results

In [ ]:
args_dict = {
        "dataloader" : dataloader,
        "model" : model,
        "optimizer" : optimizer,
        "criterion" : criterion,
        "device" : device,
        "device_ids" : None,
        "staging_device" : "cuda"
    }

In [ ]:
results = evaluate(args_dict)

<ipython-input-45-784444063ad7>:54: RuntimeWarning: divide by zero encountered in divide
  results[f"{phase}_precision"] = precision = tp / (tp + fp)
<ipython-input-45-784444063ad7>:55: RuntimeWarning: invalid value encountered in divide
  results[f"{phase}_recall"] = recall = tp / (tp + fn)
<ipython-input-45-784444063ad7>:56: RuntimeWarning: invalid value encountered in divide
  results[f"{phase}_f1"] = f1 = 2 * precision * recall / (precision + recall)


In [ ]:
results["train_accuracy"]

array([0.97386577, 0.97543383, 0.97355216, 0.08059795, 0.97543383,
       0.97459753, 0.97616559, 0.97543383, 0.97512022, 0.97480661,
       0.97543383, 0.97574744, 0.97397031, 0.97438846, 0.97585198,
       0.97480661, 0.97491114, 0.97386577, 0.97543383, 0.9756429 ,
       0.97595651, 0.97585198, 0.97491114, 0.97397031, 0.97574744,
       0.97522475, 0.97480661, 0.97512022, 0.97459753, 0.97480661,
       0.97543383, 0.97512022, 0.97407485, 0.97595651, 0.97459753,
       0.97553837, 0.97585198, 0.97532929, 0.97553837, 0.97397031])

In [ ]:
results["dev_accuracy"]

array([0.97656904, 0.97322176, 0.9790795 , 0.04769874, 0.96903766,
       0.97238494, 0.96820084, 0.97824268, 0.97656904, 0.97405858,
       0.97322176, 0.9707113 , 0.9748954 , 0.9790795 , 0.97740586,
       0.9748954 , 0.98075314, 0.97991632, 0.97740586, 0.9748954 ,
       0.96987448, 0.96820084, 0.9832636 , 0.97405858, 0.96903766,
       0.97824268, 0.97740586, 0.97322176, 0.97405858, 0.97740586,
       0.9790795 , 0.97740586, 0.97322176, 0.97322176, 0.97238494,
       0.96736402, 0.97322176, 0.97238494, 0.97824268, 0.97405858])